# sequential-adapt: Colab quickstart

Runs the test suite, a smoke test, and the **pressure grid** — the
stress-test of the replay retention result (cramming diagnostic,
replay-fraction, bigger/conflicting task families, capacity) — on a
Colab GPU. The retention grid itself is done (2026-07-14, 10 seeds:
replay fixed composed retention 0.36 -> 1.00; hard ortho inert); see
`docs/CURRENT_STATUS.md`.

**Before running:** `Runtime -> Change runtime type -> T4 GPU` (or better).

**Colab storage is ephemeral** — run the *Persist artifacts* cell at the
bottom before the VM disconnects, or your results are gone.


In [ ]:
# GPU sanity check
!nvidia-smi
import torch
print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/sequential_adapt

In [ ]:
# Clone + install (torch is preinstalled on Colab)
import os
if not os.path.isdir("sad-repo"):
    !git clone https://github.com/scottvr/sad-repo
%cd sad-repo
!git pull
!pip install -q transformers

In [ ]:
# HF setup: auth + workarounds. Re-run this cell after any runtime restart —
# env vars do not survive restarts.
import os

# Xet chunked-download backend has known hangs on Colab (downloads stall at
# ~0% forever). Plain HTTP fallback is fast when authenticated.
os.environ["HF_HUB_DISABLE_XET"] = "0"

# Auth from Colab secrets. login() persists the token to
# ~/.cache/huggingface/token, so `!python scripts/...` subprocesses pick it
# up regardless of cell order. Unauthenticated Colab IPs get throttled by
# the Hub, so this matters even for public models like distilgpt2.
# Requires "Notebook access" enabled on the HF secret (key icon, sidebar).
try:
    from google.colab import userdata
    from huggingface_hub import login
    token = userdata.get("HF_TOKEN").strip()
    login(token=token)
    print("HF token set (env + persisted to disk)")
except Exception as e:
    print(f"No HF token ({type(e).__name__}) — public models still work, "
          "but downloads may be rate-limited")

In [ ]:
# Test suite (~10 s, tiny random model; includes the new data + diagnostic tests)
!python -m pytest tests -q


In [ ]:
# Smoke test: regression check that the existing pipeline still behaves
# (~1-2 min on GPU; downloads distilgpt2 on first run)
!python scripts/smoke_test.py

In [ ]:
# Single probe before committing to the grid: a replay run now records the
# cramming diagnostic (newest vector alone on earlier tasks). Check the
# `cram (newest alone)` column / `newest_alone_on_earlier` JSON field —
# HIGH means replay's perfect retention was cramming, not repair.
!python scripts/run_controller.py --steps 200 --replay 1.0 --no-gates     --out artifacts/pressure_probe.json


## Pressure grid

Stress-tests the replay result before trusting it (reasoning in
`docs/roadmap_v0.2.md`; how to read the outcome: `docs/CURRENT_STATUS.md`).
Controller-only, `--no-gates`, 10 arms per seed:

- `frac`: replay with 1/8, 1/4, 1/2 of earlier examples
- `big`: 3 tasks x 8 facts, 12 labels, replay on/off (ceiling removed)
- `conflict`: overlapped words with conflicting labels, replay on/off
- `cap`: k = 2 / 4 / 16 with replay (k=8 covered by the retention grid)

Each `run_controller.py` call is ~70 s on an RTX 5070 Ti; budget ~3-5x
that on a T4.

- **Pilot** (next cell): 2 seeds -> 20 runs, roughly 1.5-2.5 h on a T4.
- **Full** (cell after): 5 seeds -> 50 runs — use a fast card
  (~20 min on an RTX 6000 class GPU).


In [ ]:
# Pilot grid: 2 seeds
#!SEEDS="0 1" bash scripts/run_pressure.sh

In [ ]:
# Full grid: 5 seeds (long on a T4!) — uncomment to run
!bash scripts/run_pressure.sh


## Render Summary

In [ ]:
# Aggregate and render the summary inline
!python scripts/summarize_sweeps.py
from IPython.display import Markdown
Markdown(open("artifacts/sweeps_summary.md").read())

## Persist artifacts (run before the VM disconnects)

In [ ]:
# Option A: zip + browser download
!zip -qr artifacts.zip artifacts
from google.colab import files
files.download("artifacts.zip")

Reading the table: condition labels are inferred from config fields —
`replay=1;frac=0.25`, `replay=1;overlap=2`, `replay=1;tasks=3x8;labels=12`,
`replay=1;k=2`, ... **Read the `cram (newest alone)` column before believing
any retention number**: high (~earlier-task accuracy) means the newest vector
re-learned everything alone; low means genuine composition repair. Full
interpretation guide: `docs/CURRENT_STATUS.md` (How to read the outcome).


In [ ]:
!cp -r artifacts /content/drive/MyDrive/sequential_adapt/